# Etapa 1

In [ ]:
# dados disponíveis em: https://download.inep.gov.br/microdados/microdados_enem_2025.zip
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('ENEM-SPARK-STUDY')
    .master('local[4]')
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "America/Sao_Paulo")
    .getOrCreate()
)

c:\Users\Usuario\Desktop\data-science\enem-spark-study\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
df = spark.read.csv('../data/bronze/RESULTADOS_2025.csv', sep=';', encoding='iso-8859-1', header=True, inferSchema=True)
df.printSchema()

root
 |-- NU_SEQUENCIAL: integer (nullable = true)
 |-- NU_ANO: integer (nullable = true)
 |-- CO_ESCOLA: integer (nullable = true)
 |-- CO_MUNICIPIO_ESC: integer (nullable = true)
 |-- NO_MUNICIPIO_ESC: string (nullable = true)
 |-- CO_UF_ESC: integer (nullable = true)
 |-- SG_UF_ESC: string (nullable = true)
 |-- TP_DEPENDENCIA_ADM_ESC: integer (nullable = true)
 |-- TP_LOCALIZACAO_ESC: integer (nullable = true)
 |-- TP_SIT_FUNC_ESC: integer (nullable = true)
 |-- CO_MUNICIPIO_PROVA: integer (nullable = true)
 |-- NO_MUNICIPIO_PROVA: string (nullable = true)
 |-- CO_UF_PROVA: integer (nullable = true)
 |-- SG_UF_PROVA: string (nullable = true)
 |-- TP_PRESENCA_CN: integer (nullable = true)
 |-- TP_PRESENCA_CH: integer (nullable = true)
 |-- TP_PRESENCA_LC: integer (nullable = true)
 |-- TP_PRESENCA_MT: integer (nullable = true)
 |-- CO_PROVA_CN: integer (nullable = true)
 |-- CO_PROVA_CH: integer (nullable = true)
 |-- CO_PROVA_LC: integer (nullable = true)
 |-- CO_PROVA_MT: integer 

In [3]:
df.count()

4810772

In [4]:
df.describe().show()

+-------+------------------+-------+--------------------+------------------+----------------+------------------+---------+----------------------+-------------------+---------------+------------------+------------------+------------------+-----------+------------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+---------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+------------------+-------

In [5]:
df.show()

+-------------+------+---------+----------------+-----------------+---------+---------+----------------------+------------------+---------------+------------------+-------------------+-----------+-----------+--------------+--------------+--------------+--------------+-----------+-----------+-----------+-----------+----------+----------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------+-------------+-------------+-------------+-------------+---------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+----

In [6]:
df.select(['TP_PRESENCA_CN'])

DataFrame[TP_PRESENCA_CN: int]

In [7]:
# verificando valores nulos
from pyspark.sql.functions import col, sum, when

columns = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT','TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT']

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in columns
]).show()

+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|NU_NOTA_CN|NU_NOTA_CH|NU_NOTA_LC|NU_NOTA_MT|TP_PRESENCA_CN|TP_PRESENCA_CH|TP_PRESENCA_LC|TP_PRESENCA_MT|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|   1550436|   1353217|   1353217|   1550436|             0|             0|             0|             0|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+



In [8]:
# verificando valores nulos por aqueles que compareceram no primeiro dia ou seja (1 = presença, 2 = eliminado), pegando ou LC ou CH que são do primeiro dia
from pyspark.sql.functions import col, sum, when

columns = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT','TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT']

(df
 .filter(col("TP_PRESENCA_CH") == 1)
 .select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in columns])
 .show()
)

+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|NU_NOTA_CN|NU_NOTA_CH|NU_NOTA_LC|NU_NOTA_MT|TP_PRESENCA_CN|TP_PRESENCA_CH|TP_PRESENCA_LC|TP_PRESENCA_MT|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|    213207|         0|         0|    213207|             0|             0|             0|             0|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+



In [9]:
# verificando valores nulos por aqueles que compareceram no primeiro dia ou seja (1 = presença, 2 = eliminado), pegando agora CN ou MT que são dos segundo dia
from pyspark.sql.functions import col, sum, when

columns = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT','TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT']

(df
 .filter(col("TP_PRESENCA_CN") == 1)
 .select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in columns])
 .show()
)

+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|NU_NOTA_CN|NU_NOTA_CH|NU_NOTA_LC|NU_NOTA_MT|TP_PRESENCA_CN|TP_PRESENCA_CH|TP_PRESENCA_LC|TP_PRESENCA_MT|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|         0|     15988|     15988|         0|             0|             0|             0|             0|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+



In [10]:
# verificando valores nulos por aqueles que compareceram no primeiro dia ou seja (1 = presença, 2 = eliminado), pegando agora CN ou MT que são dos segundo dia
from pyspark.sql.functions import col, sum, when

columns = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT','TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT']

(df
 .filter(col("TP_PRESENCA_LC") == 1)
 .count()
)

3457555

Temos um total de 4810772 registros, com a grande maioria sendo inferida através da utilização do inferSchema na leitura do csv.

- Dentre as colunas de notas, temos aproximadamente de 28-31% de valores NULOS, sendo que não temos valores nulos nas colunas de presença.
- Pessoas que compareceram no primeiro dia mas não no segundo: 213207
- Pessoas que compareceram no segundo dia mas não no primeiro: 15988 (que é esperado ser menor mesmo já que candidatos não tem mais chance de passar na grande maioria dos cursos)

-> Ao compararmos os valores de ausência, vemos que bate exatamente com os valores nulos de notas em seus respectivos dias, com o segundo dia tendo mais faltas que o primeiro conforme esperado. Podemos deduzir então que os valores nulos são exatamente as candidatos que faltaram essas provas mas tem inscrição.

# Etapa 2

In [11]:
# Filtrar apenas candidatos da Paraíba (SG_UF_PROVA = 'PB')
from pyspark.sql.functions import col

df_pb = df.filter(col("SG_UF_PROVA") == 'PB')

df_pb.show()

+-------------+------+---------+----------------+----------------+---------+---------+----------------------+------------------+---------------+------------------+--------------------+-----------+-----------+--------------+--------------+--------------+--------------+-----------+-----------+-----------+-----------+----------+----------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------+-------------+-------------+-------------+-------------+---------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+----

In [12]:
# Filtrar apenas candidatos presentes em todas as provas (TP_PRESENCA_* = 1)
from pyspark.sql.functions import col

df_pb_presenca = (df_pb
 .filter(
    (col("TP_PRESENCA_CN") == 1) &
    (col("TP_PRESENCA_CH") == 1) &
    (col("TP_PRESENCA_LC") == 1) &
    (col("TP_PRESENCA_MT") == 1)
))

df_pb_presenca.show()

+-------------+------+---------+----------------+----------------+---------+---------+----------------------+------------------+---------------+------------------+--------------------+-----------+-----------+--------------+--------------+--------------+--------------+-----------+-----------+-----------+-----------+----------+----------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------+-------------+-------------+-------------+-------------+---------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+----

In [13]:
display(df.count())
display(df.count() - df_pb.count())
display(df_pb.count())
display(df_pb.count() - df_pb_presenca.count())
display(df_pb_presenca.count())

4810772

4668737

142035

41152

100883

Aqui vemos que tivemos 4668737 registros descartados no primeiro filtro por conta de termos selecionado apenas um dos 27 estados, sendo que a Paraíba não é um dos estados com maiores quantidades de pessoas (tipo São Paulo).

Enquanto isso, no filtro de presença, vemos que corresponde a (41152/142035) * 100 que é praticamente 29% das pessoas que faltaram a prova, correspondendo a nosso intervalo geral de 28-31% de pessoas que faltaram somando todos os estados.

In [14]:
# Criar coluna NOTA_MEDIA
from pyspark.sql.functions import col

df = df.withColumn('NOTA_MEDIA', (
    col('NU_NOTA_CH') + 
    col('NU_NOTA_LC') + 
    col('NU_NOTA_MT') + 
    col('NU_NOTA_CN')
    )
    /4
)

df.select('NU_SEQUENCIAL', 'NOTA_MEDIA').show()

+-------------+------------------+
|NU_SEQUENCIAL|        NOTA_MEDIA|
+-------------+------------------+
|       206403| 584.1999999999999|
|      3604651|              NULL|
|      1461268|           413.875|
|      4301058|              NULL|
|      3148322|             540.0|
|      1035588|             516.3|
|      3587454|            426.95|
|       140768|             549.5|
|      3541867|             621.0|
|      3371353|           433.825|
|       377894|             434.8|
|      3446319|              NULL|
|       134650|             545.8|
|      1620924|              NULL|
|       588106|              NULL|
|      1399198|424.07500000000005|
|       941795|           458.375|
|      1894153|379.27500000000003|
|      2262755|              NULL|
|      3575012|           588.125|
+-------------+------------------+
only showing top 20 rows


In [15]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "FAIXA_DESEMPENHO",
    when(col("NOTA_MEDIA").isNull(), None)
    .when(col("NOTA_MEDIA") < 400, "ABAIXO DO BÁSICO")
    .when(col("NOTA_MEDIA") < 550, "BÁSICO")
    .when(col("NOTA_MEDIA") < 700, "ADEQUADO")
    .otherwise("AVANÇADO")
)

df.select("NOTA_MEDIA", "FAIXA_DESEMPENHO").show()

+------------------+----------------+
|        NOTA_MEDIA|FAIXA_DESEMPENHO|
+------------------+----------------+
| 584.1999999999999|        ADEQUADO|
|              NULL|            NULL|
|           413.875|          BÁSICO|
|              NULL|            NULL|
|             540.0|          BÁSICO|
|             516.3|          BÁSICO|
|            426.95|          BÁSICO|
|             549.5|          BÁSICO|
|             621.0|        ADEQUADO|
|           433.825|          BÁSICO|
|             434.8|          BÁSICO|
|              NULL|            NULL|
|             545.8|          BÁSICO|
|              NULL|            NULL|
|              NULL|            NULL|
|424.07500000000005|          BÁSICO|
|           458.375|          BÁSICO|
|379.27500000000003|ABAIXO DO BÁSICO|
|              NULL|            NULL|
|           588.125|        ADEQUADO|
+------------------+----------------+
only showing top 20 rows


In [80]:
from pyspark.sql.functions import col

df_silver = (
    df
    .filter(col("SG_UF_ESC") == "PB")
    .select(
        'NO_MUNICIPIO_ESC',          # nome do município da escola
        'NO_MUNICIPIO_PROVA',        # município onde realizou a prova
        'TP_DEPENDENCIA_ADM_ESC',    # tipo de escola
        'TP_LOCALIZACAO_ESC',        # localização da escola
        'TP_LINGUA',                 # língua estrangeira
        'TP_PRESENCA_CN',            # presença em Ciências da Natureza
        'TP_PRESENCA_CH',            # presença em Ciências Humanas
        'TP_PRESENCA_LC',            # presença em Linguagens
        'TP_PRESENCA_MT',            # presença em Matemática
        'NU_NOTA_CN',                # nota de Ciências da Natureza
        'NU_NOTA_CH',                # nota de Ciências Humanas
        'NU_NOTA_LC',                # nota de Linguagens
        'NU_NOTA_MT',                # nota de Matemática
        'NU_NOTA_REDACAO',           # nota de Redação
        'NU_NOTA_COMP1',             # nota media da competencia 1 da redação do candidato
        'NOTA_MEDIA',                # média das quatro provas objetivas
        'FAIXA_DESEMPENHO'           # faixa de desempenho
    )
)

In [17]:
df_silver.show()

+----------------+--------------------+----------------------+---------+----------+----------+----------+----------+---------------+------------------+----------------+
|NO_MUNICIPIO_ESC|  NO_MUNICIPIO_PROVA|TP_DEPENDENCIA_ADM_ESC|TP_LINGUA|NU_NOTA_CN|NU_NOTA_CH|NU_NOTA_LC|NU_NOTA_MT|NU_NOTA_REDACAO|        NOTA_MEDIA|FAIXA_DESEMPENHO|
+----------------+--------------------+----------------------+---------+----------+----------+----------+----------+---------------+------------------+----------------+
|     João Pessoa|         João Pessoa|                     4|        0|     591.9|     593.3|     624.8|     674.0|            820|             621.0|        ADEQUADO|
|     João Pessoa|         João Pessoa|                     4|        0|     445.4|     439.9|     473.8|     407.2|            640|441.57500000000005|          BÁSICO|
|            NULL|         João Pessoa|                  NULL|        0|     517.9|     625.7|     578.3|     508.0|            920|           557.475|    

In [43]:
# salvando em parquet
path = '../data/silver/municipio_prova'

(
    df_silver
    .write
    .mode('overwrite')
    .partitionBy('NO_MUNICIPIO_PROVA')
    .parquet(path)
)

In [19]:
df.explain()

== Physical Plan ==
*(1) Project [NU_SEQUENCIAL#23, NU_ANO#24, CO_ESCOLA#25, CO_MUNICIPIO_ESC#26, NO_MUNICIPIO_ESC#27, CO_UF_ESC#28, SG_UF_ESC#29, TP_DEPENDENCIA_ADM_ESC#30, TP_LOCALIZACAO_ESC#31, TP_SIT_FUNC_ESC#32, CO_MUNICIPIO_PROVA#33, NO_MUNICIPIO_PROVA#34, CO_UF_PROVA#35, SG_UF_PROVA#36, TP_PRESENCA_CN#37, TP_PRESENCA_CH#38, TP_PRESENCA_LC#39, TP_PRESENCA_MT#40, CO_PROVA_CN#41, CO_PROVA_CH#42, CO_PROVA_LC#43, CO_PROVA_MT#44, NU_NOTA_CN#45, NU_NOTA_CH#46, NU_NOTA_LC#47, ... 47 more fields]
+- *(1) Project [NU_SEQUENCIAL#23, NU_ANO#24, CO_ESCOLA#25, CO_MUNICIPIO_ESC#26, NO_MUNICIPIO_ESC#27, CO_UF_ESC#28, SG_UF_ESC#29, TP_DEPENDENCIA_ADM_ESC#30, TP_LOCALIZACAO_ESC#31, TP_SIT_FUNC_ESC#32, CO_MUNICIPIO_PROVA#33, NO_MUNICIPIO_PROVA#34, CO_UF_PROVA#35, SG_UF_PROVA#36, TP_PRESENCA_CN#37, TP_PRESENCA_CH#38, TP_PRESENCA_LC#39, TP_PRESENCA_MT#40, CO_PROVA_CN#41, CO_PROVA_CH#42, CO_PROVA_LC#43, CO_PROVA_MT#44, NU_NOTA_CN#45, NU_NOTA_CH#46, NU_NOTA_LC#47, ... 46 more fields]
   +- FileScan cs

Nesse explain, o spark está lendo o csv, depois criando a nova coluna NOTA_MEDIA, e logo em seguida criando outra coluna FAIXA_DESEMPENHO

# Etapa 3

In [40]:
df_silver.filter(col('NO_MUNICIPIO_PROVA') == 'Maringá').show()

+----------------+------------------+----------------------+---------+----------+----------+----------+----------+---------------+----------+----------------+
|NO_MUNICIPIO_ESC|NO_MUNICIPIO_PROVA|TP_DEPENDENCIA_ADM_ESC|TP_LINGUA|NU_NOTA_CN|NU_NOTA_CH|NU_NOTA_LC|NU_NOTA_MT|NU_NOTA_REDACAO|NOTA_MEDIA|FAIXA_DESEMPENHO|
+----------------+------------------+----------------------+---------+----------+----------+----------+----------+---------------+----------+----------------+
+----------------+------------------+----------------------+---------+----------+----------+----------+----------+---------------+----------+----------------+



In [ ]:
# pergunta 1: vendo as 10 maiores notas medias por municipio e as 10 menores notas medias
from pyspark.sql.functions import col
from pyspark.sql.functions import avg

(
    df_silver
    .filter(col('NO_MUNICIPIO_ESC').isNotNull() & col('NOTA_MEDIA').isNotNull())
    .groupBy('NO_MUNICIPIO_ESC')
    .agg(avg('NOTA_MEDIA').alias('NOTA_MEDIA_MUNICIPIO'))
    .orderBy(col('NOTA_MEDIA_MUNICIPIO').desc())
    .show(10)
)

+----------------+--------------------+
|NO_MUNICIPIO_ESC|NOTA_MEDIA_MUNICIPIO|
+----------------+--------------------+
|     João Pessoa|   534.2952271355269|
|  Campina Grande|    527.836157147085|
| Catolé do Rocha|   516.9304268292683|
|       Guarabira|   514.8423145933015|
|      Itaporanga|   513.9362654320988|
|           Patos|   512.7559772296015|
|           Sousa|   512.4077288941735|
|           Picuí|   511.0503311258279|
|          Pombal|  510.74365558912376|
|          Parari|  510.55961538461537|
+----------------+--------------------+
only showing top 10 rows


In [46]:
# 10 menores agora
(
    df_silver
    .filter(col('NO_MUNICIPIO_ESC').isNotNull() & col('NOTA_MEDIA').isNotNull())
    .groupBy('NO_MUNICIPIO_ESC')
    .agg(avg('NOTA_MEDIA').alias('NOTA_MEDIA_MUNICIPIO'))
    .orderBy(col('NOTA_MEDIA_MUNICIPIO').asc())
    .show(10)
)

+--------------------+--------------------+
|    NO_MUNICIPIO_ESC|NOTA_MEDIA_MUNICIPIO|
+--------------------+--------------------+
|             Quixaba|   413.4846153846154|
|   Areia de Baraúnas|  432.65714285714284|
|              Zabelê|  432.71666666666675|
|  São José de Caiana|  434.70833333333326|
|            Gurinhém|  435.20471698113204|
|  São José dos Ramos|   435.4206896551724|
|            Marcação|   436.2010638297873|
|São José do Brejo...|   436.7264705882352|
|   São João do Tigre|  437.22875000000005|
|                Emas|   440.9333333333333|
+--------------------+--------------------+
only showing top 10 rows


Aqui vemos que no geral os 10 municipios com maior nota média estão concentrados mais nas grandes cidades, mais próximas do litoral como JP e CG, enquanto os 10 municipios com menor nota média são mais interioranos, pro sertão e etc. mostrando uma desigualdade clara no estado da Paraíba, onde estudantes que estudam nessas cidades maiores tendem a ter maiores notas médias no ENEM

In [ ]:
# pergunta 2: relação entre escolha da lingua estrangeira e nota media de linguagens e codigos

(
    df_silver
    .groupBy('TP_LINGUA')
    .agg(avg('NU_NOTA_LC').alias('NOTA_MEDIA_LC'))
    .show()
)

+---------+-----------------+
|TP_LINGUA|    NOTA_MEDIA_LC|
+---------+-----------------+
|        1|491.1624830894107|
|        0|538.1773339867678|
+---------+-----------------+



Aqui vemos que as pessoas que escolhem inglês (codigo 0) costumam ter uma nota média em linguas e códigos aproximadamente 5% maior do que as que escolhem espanhol (codigo 1)

In [74]:
# pergunta 3: calculando taxa de abstenção, vamos usar aqui uma coluna de presenca_geral, que é quando o candidato está presente nas 4 provas

df_silver = (
    df_silver
    .withColumn(
        'PRESENCA_GERAL',
        when(
            (col('TP_PRESENCA_MT') == 1) &
            (col('TP_PRESENCA_CN') == 1) &
            (col('TP_PRESENCA_LC') == 1) &
            (col('TP_PRESENCA_MT') == 1),
            1            
        ).otherwise(0)
)
)

In [57]:
display(
    df_silver
    .agg((100 * avg('PRESENCA_GERAL')).alias('PRESENCA_GERAL'))
    .show()
)

display(
    df_silver
    .groupby('TP_DEPENDENCIA_ADM_ESC')
    .agg((100 * avg('PRESENCA_GERAL')).alias('PRESENCA_GERAL'))
    .orderBy(col('TP_DEPENDENCIA_ADM_ESC').asc())
    .show()
)

+-----------------+
|   PRESENCA_GERAL|
+-----------------+
|79.51410697686104|
+-----------------+



None

+----------------------+------------------+
|TP_DEPENDENCIA_ADM_ESC|    PRESENCA_GERAL|
+----------------------+------------------+
|                     1| 96.63658736669402|
|                     2| 74.86804451510334|
|                     3|58.139534883720934|
|                     4| 97.44550408719346|
+----------------------+------------------+



None

- 1 Federal
- 2 Estadual
- 3 Municipal
- 4 Privada

Como temos essa relação, vemos que a privada tem a maior taxa de presença (97,44%), enquanto a municipal possui a menor taxa (58,13%), o que revela um pouco da desigualdade e dificuldades

In [ ]:
# pergunta 4: diferença entre notas medias usando TP_DEPENDENCIA_ADM_ESC (- 1 Federal - 2 Estadual - 3 Municipal - 4 Privada)
(
    df_silver
    .groupBy("TP_DEPENDENCIA_ADM_ESC")
    .agg(
        avg("NU_NOTA_CN").alias("MEDIA_CN"),
        avg("NU_NOTA_CH").alias("MEDIA_CH"),
        avg("NU_NOTA_LC").alias("MEDIA_LC"),
        avg("NU_NOTA_MT").alias("MEDIA_MT"),
        avg("NU_NOTA_REDACAO").alias("MEDIA_REDACAO"),
        avg("NOTA_MEDIA").alias("MEDIA_GERAL")
    )
    .orderBy("TP_DEPENDENCIA_ADM_ESC")
    .show()
)

+----------------------+------------------+------------------+------------------+------------------+-----------------+------------------+
|TP_DEPENDENCIA_ADM_ESC|          MEDIA_CN|          MEDIA_CH|          MEDIA_LC|          MEDIA_MT|    MEDIA_REDACAO|       MEDIA_GERAL|
+----------------------+------------------+------------------+------------------+------------------+-----------------+------------------+
|                     1| 525.8531806615777| 538.2691108301727|  552.593215339233| 568.1602205258694|698.4745048461863| 546.3375318336164|
|                     2| 473.0157700935373|474.63829830813984| 498.0834951456312|463.77765691793286|521.0159354389415| 478.0839367620825|
|                     3|461.00800000000004| 455.1714285714285|476.11428571428576|417.02799999999996|360.7142857142857|450.05299999999994|
|                     4|   550.15007857517|  561.973709733287| 569.6466920678904|  611.630923694779|756.5985452026325| 573.7840833624607|
+----------------------+----------

Vemos que candidatos de escolas públicas (1,2,3) tem menor nota_media geral do que candidatos de escolas privadas (4), revelando desigualdade no estado da Paraíba, essa diferença se mantém em todas as áreas do conhecimento, onde a privada (4) sempre tem media maior que as outras

In [ ]:
# pergunta 5: qual o percentual de candidatos nota 0? e com nota 1000? existe correlação entre notas extremas e TP_DEPENDENCIA_ADM_ESC?

# percentual de 0:

(
    df_silver
    .withColumn('NOTA_0', when(col('NU_NOTA_REDACAO') == 0, 1).otherwise(0))
    .agg((100 * avg('NOTA_0')).alias('NOTA_0'))
    .show()
)

+-----------------+
|           NOTA_0|
+-----------------+
|8.793307037158003|
+-----------------+



In [70]:
#  percentual de 1000:
(
    df_silver
    .withColumn('NOTA_1000', when(col('NU_NOTA_REDACAO') == 1000, 1).otherwise(0))
    .agg((100 * avg('NOTA_1000')).alias('NOTA_1000'))
    .show()
)

+---------+
|NOTA_1000|
+---------+
|      0.0|
+---------+



In [71]:
(
    df_silver
    .withColumn('NOTA_0', when(col('NU_NOTA_REDACAO') == 0, 1).otherwise(0))
    .groupBy('TP_DEPENDENCIA_ADM_ESC')
    .agg((100 * avg('NOTA_0')).alias('NOTA_0'))
    .orderBy('TP_DEPENDENCIA_ADM_ESC')
    .show()
)

+----------------------+------------------+
|TP_DEPENDENCIA_ADM_ESC|            NOTA_0|
+----------------------+------------------+
|                     1|3.0352748154224773|
|                     2|10.585055643879175|
|                     3|  18.6046511627907|
|                     4|1.5156675749318802|
+----------------------+------------------+



Como não tivemos nota 1000, analisamos somente os 8% (aproximadamente) que tiraram nota 0, daí vemos que as maiores quantidades estão em escolas estaduais e municipais, enquanto as privadas tem menor quantidade

### Pergunta original 1: Comparar a taxa de abstenção por localização da escola e tipo de dependência administrativa

Essa pergunta se justifica pois o estado pode ver se a quantidade de faltas tem relação com a localização da escola, portanto pode investir melhor na separação de locais de prova por localização de escolas e etc.

- 1: Urbana
- 2: Rural

In [76]:
(
    df_silver
    .groupBy('TP_LOCALIZACAO_ESC')
    .agg(avg('PRESENCA_GERAL').alias('PRESENCA_GERAL'))
    .orderBy('TP_LOCALIZACAO_ESC')
    .show()
)

+------------------+------------------+
|TP_LOCALIZACAO_ESC|    PRESENCA_GERAL|
+------------------+------------------+
|                 1|0.7994087174925436|
|                 2| 0.691967109424415|
+------------------+------------------+



In [78]:
(
    df_silver
    .groupBy('TP_LOCALIZACAO_ESC', 'TP_DEPENDENCIA_ADM_ESC')
    .agg(avg('PRESENCA_GERAL').alias('PRESENCA_GERAL'))
    .orderBy('TP_LOCALIZACAO_ESC', 'TP_DEPENDENCIA_ADM_ESC')
    .show()
)

+------------------+----------------------+------------------+
|TP_LOCALIZACAO_ESC|TP_DEPENDENCIA_ADM_ESC|    PRESENCA_GERAL|
+------------------+----------------------+------------------+
|                 1|                     1|0.9664570230607966|
|                 1|                     2|0.7520710849812934|
|                 1|                     3|0.4827586206896552|
|                 1|                     4|0.9744550408719346|
|                 2|                     1|0.9622641509433962|
|                 2|                     2|0.6816380449141347|
|                 2|                     3|0.7857142857142857|
+------------------+----------------------+------------------+



Interessantemente, vemos que não existe escola privada no âmbito rural (combinação 2,4), e as escolas municipais que são urbanas (1,3) tem maior taxa de presença até mesmo do que as municipais que estão no campo rural (2,3), então isso pode mostrar que essas escolas municipais urbanas podem estar mais distantes que as que são rurais dos candidatos, e por serem de baixa renda não tem como chegar até essas escolas (uma das hipóteses), então poderia ser investido uma maior infraestrutura para esses candidatos municipais da cidade urbana

### Pergunta 2: Existe diferença no desempenho da Competência 1 da redação entre alunos de escolas públicas e privadas?

A Competência 1 avalia o domínio da modalidade escrita formal da Língua Portuguesa, refletindo o conhecimento gramatical, ortográfico e sintático dos participantes. Comparar essa competência entre os diferentes tipos de dependência administrativa da escola permite identificar possíveis desigualdades na formação linguística dos estudantes.

Essa análise pode auxiliar o estado a verificar se existe uma discrepância significativa no domínio da norma padrão entre alunos de escolas federais, estaduais, municipais e privadas, fornecendo evidências para a formulação de políticas públicas voltadas ao fortalecimento do ensino de Língua Portuguesa.

- 1 Federal
- 2 Estadual
- 3 Municipal
- 4 Privada

In [81]:
(
    df_silver
    .groupBy('TP_DEPENDENCIA_ADM_ESC')
    .agg(avg('NU_NOTA_COMP1').alias('NOTA_MEDIA_GRAMATICA'))
    .orderBy('TP_DEPENDENCIA_ADM_ESC')
    .show()
)

+----------------------+--------------------+
|TP_DEPENDENCIA_ADM_ESC|NOTA_MEDIA_GRAMATICA|
+----------------------+--------------------+
|                     1|  128.72313527180785|
|                     2|   99.02666830527221|
|                     3|                70.0|
|                     4|  137.83858676827157|
+----------------------+--------------------+



Como esperado, temos as piores notas em relação a gramática para escolas municipais (70), enquanto as privadas tem maiores notas para gramática (137), o que sugere que um dos problemas a serem resolvidos para aumentar a nota da redação é melhorar a qualidade do ensino na gramática em escolas municipais, o que vai de acordo com algumas analises anteriores

# Etapa 4

- tive que instalar o UV primeiro na minha máquina, depois usei o uv add com as dependencias, logo em seguida instalei o java 17 com openjdk
- decidi usar shuffle com 8 partições por ser algo mais local, além de 4 cores para evitar sobrecargas no processador
- da primeira vez que li o csv ele só identificou uma coluna por conta de eu não ter usado o ; como separador de colunas
- após ajeitar o ;, por ser um csv, ele só leu como string o tipo das colunas, daí foi necessário ajeitar essa tipagem com inferSchema = True na leitura
- foi necessário inserir o encoding em latin (iso8859-1) além de header=True para identificar esse cabeçalho
- ao tentar fazer a parte da faixa_desempenho usando UDF obtive erro de timeout no worker python, tive dificuldade nessa parte então fiz com when e otherwise para otimizar com catalyst
- tive dificuldade para gerar os parquet no windows, tive que adicionar as utils do hadoop no PATH para dar certo